# Pipeline QR finale — vue de bout en bout

Ce notebook montre la chaîne réelle ayant mené au QR final E040 : **QR exact → condition ControlNet → Stage 1 → Stage 2 → SR-MPGD i0…i8 → sélection du meilleur checkpoint → QR final**.

Il affiche aussi le modèle advisor E026/E031 et le surrogate E016 avec leur rôle exact. Le parent Stage 1/Stage 2 reste figé pour que l'optimisation E040 soit comparable à E035–E039.


In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import Image as DisplayImage, Markdown, display
RESULTS_DIR=Path(os.environ.get('E040_RESULTS_DIR','/data/e040-srmpgd-checkpoint-frontier-v1'))
manifest=json.loads((RESULTS_DIR/'pipeline-manifest.json').read_text(encoding='utf-8'))
verdict=json.loads((RESULTS_DIR/'verdict.json').read_text(encoding='utf-8'))
display(Markdown(f"**Payload :** `{manifest['payload']}`  \
**Prompt :** {manifest['prompt']}  \
**γ :** {verdict['gamma']}"))


## 1. La pipeline entière

In [ ]:
display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/full-pipeline-contact-sheet.png'),width=1500))


## 2. QR exact / condition ControlNet

In [ ]:
display(Markdown('### QR exact payload')); display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/01-qr-reference.png'),width=600))
display(Markdown('### Condition binaire envoyée au ControlNet')); display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/02-control-condition.png'),width=700))


## 3. Stage 1 — diffusion artistique

In [ ]:
p=RESULTS_DIR/'pipeline/03-stage1.png'
if p.is_file(): display(DisplayImage(filename=str(p),width=760))
else: display(Markdown('Stage 1 archivé absent dans cette image de déploiement.'))


## 4. Stage 2 — SRPG

In [ ]:
display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/04-stage2.png'),width=760))


## 5. SR-MPGD — tous les checkpoints

In [ ]:
df=pd.read_csv(RESULTS_DIR/'checkpoint-comparison.csv')
recipe=verdict['research_winner_recipe']; selected=int(verdict['winner_iteration'])
for i in range(9):
 row=df[(df.method==recipe)&(df.iteration==i)].iloc[0]
 label=' ⭐ FINAL' if i==selected else ''
 display(Markdown(f"### SR-MPGD i{i}{label} — SSR={int(row.qr_verify_exact_presets)}/37 · MER={int(row.full_module_error_count)}/841 · LPIPS={row.lpips:.4f} · safe={row.visual_guard_pass}"))
 display(DisplayImage(filename=str(RESULTS_DIR/recipe/'images'/f'iteration-{i:03d}.png'),width=760))


## 6. QR FINAL sélectionné

In [ ]:
display(DisplayImage(filename=str(RESULTS_DIR/'pipeline/99-FINAL-QR.png'),width=850))
display(verdict)


## 7. Pourquoi ce checkpoint a gagné ?

In [ ]:
raw=json.loads((RESULTS_DIR/'checkpoint-comparison.json').read_text(encoding='utf-8'))
w=next(x for x in raw if x['checkpoint']==verdict['research_winner_checkpoint'])
display(pd.DataFrame([{'metric':'SSR /37','value':w['qr_verify_exact_presets']},{'metric':'original_exact','value':w['original_exact']},{'metric':'MER /841','value':w['full_module_error_count']},{'metric':'LPIPS','value':w['lpips']},{'metric':'latent RMS','value':w['latent_delta_rms']},{'metric':'CLIPScore','value':w.get('clip_score')},{'metric':'CLIP-Aesthetic','value':w.get('clip_aesthetic')},{'metric':'HPS','value':w.get('hpsv2_1')},{'metric':'E016 mean p(scan)','value':w.get('surrogate_mean_success_probability')},{'metric':'visual safe','value':w['visual_guard_pass']}]))
display(Markdown('### Détail de la garde visuelle')); display(w.get('visual_guard_checks'))


## 8. Modèle advisor E026/E031

In [ ]:
advisor=json.loads((RESULTS_DIR/'advisor-preview.json').read_text(encoding='utf-8'))
if advisor.get('available'):
 display(Markdown('Le modèle est chargé et recommande des paramètres **avant génération**. Il ne certifie jamais le QR.'))
 display(pd.DataFrame(advisor.get('recommendations',[])))
else: display(advisor)


## 9. Modèle E016 différentiable

In [ ]:
status=json.loads((RESULTS_DIR/'e016-surrogate-status.json').read_text(encoding='utf-8')); display(status)
if status.get('research_usable'):
 display(Markdown('E016 score les checkpoints en plus des vrais décodeurs. Dans E040 il est volontairement **secondaire à QR-Verify**.'))


## 10. Carte finale de la chaîne

In [ ]:
display(Markdown('''```text
Prompt + payload
      ↓
Advisor E026/E031 (recommandation)
      ↓
QR exact → condition QR Monster
      ↓
Stage 1 : Cetus-Mix + ControlNet
      ↓
Stage 2 : SRPG
      ↓ latent exact z0
SR-MPGD scan-aware, γ=1000
      ↓ i0..i8
QR-Verify + garde esthétique + score E016
      ↓
MEILLEUR CHECKPOINT SÛR
      ↓
99-FINAL-QR.png
```'''))
